<a href="https://colab.research.google.com/github/Biel07-max/Projeto-analise-de-dados/blob/main/sistema_de_analise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Painel COVID-19 – Conitec (POO)

Fluxo do projeto:

```
Fonte (CSV/XLS) ──► TratadorDados ──► AnalisadorDados ──► Dashboard / Visualizador
   carregar()          limpar()          limpar()            exibir_kpis()
                                         calcular_indicadores()   exibir_graficos()
                                                                  exibir_tabela()
```

| Classe | Papel |
|---|---|
| `FonteDados` / `FonteCSV` / `FonteXLS` | carregar os dados brutos |
| `TratadorDados` | limpeza e filtros genéricos |
| `AnalisadorDados` | regras específicas da Conitec (usa o `TratadorDados`) e cálculo dos `Indicador`es |
| `Indicador` | um KPI com formatação no padrão brasileiro |
| `Visualizador` | gráficos |
| `Dashboard` | monta o painel (KPIs, gráficos e tabela) |

Ordem de execução: instalar/importar → classes de apoio → carregar e tratar os dados → `Visualizador` → `Dashboard` → executar → filtros.

In [54]:
!pip -q install plotly openpyxl ipywidgets

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
import re
import io
from IPython.display import display, HTML, clear_output

## Classes de apoio: fontes de dados e `Indicador`

In [55]:
# ============================================================
# FONTES DE DADOS (herança + polimorfismo)
# ============================================================
class FonteDados:
    """Classe base: toda fonte precisa implementar carregar()."""

    def __init__(self, nome):
        self.nome = nome
        self._dados = None

    @property
    def dados(self):
        return self._dados

    def carregar(self):
        raise NotImplementedError("Implemente este método na classe filha")


class FonteCSV(FonteDados):
    def __init__(self, nome, caminho, sep=",", encoding="utf-8"):
        super().__init__(nome)
        self.caminho = caminho
        self.sep = sep
        self.encoding = encoding

    def carregar(self):
        self._dados = pd.read_csv(self.caminho, sep=self.sep, encoding=self.encoding)
        return self._dados


class FonteXLS(FonteDados):
    def __init__(self, nome, caminho, aba=0):
        super().__init__(nome)
        self.caminho = caminho
        self.aba = aba

    def carregar(self):
        self._dados = pd.read_excel(self.caminho, sheet_name=self.aba)
        return self._dados


# ============================================================
# INDICADOR (KPI)
# ============================================================
class Indicador:
    """Representa um indicador (KPI)."""

    def __init__(self, nome, valor, unidade="", descricao="", casas=2):
        self.nome = nome
        self.unidade = unidade
        self.descricao = descricao
        self.casas = casas          # casas decimais (0 para contagens)
        self._valor = valor

    @property
    def valor(self):
        return self._valor

    def formatar(self):
        texto = f"{self.valor:,.{self.casas}f}"
        # padrão brasileiro: 1.234,56
        texto = texto.replace(",", "X").replace(".", ",").replace("X", ".")
        if self.unidade == "R$":
            return f"R$ {texto}"
        return f"{texto} {self.unidade}".strip()

    def __str__(self):
        return f"{self.nome}: {self.formatar()}"

## Tratamento e análise: `TratadorDados` e `AnalisadorDados`

In [56]:
# ============================================================
# TRATADOR (limpeza e filtros genéricos)
# ============================================================
class TratadorDados:
    """Responsável por limpar e filtrar os dados."""

    def __init__(self, dataframe):
        self.__df = dataframe.copy()          # privado: trabalhamos numa cópia

    @property
    def df(self):
        return self.__df

    # ---------- tratamento (altera o objeto e devolve self) ----------
    def limpar(self):
        """'-' vira ausente, some linha vazia, some duplicado, textos sem espaço sobrando."""
        df = self.__df.replace(r"^\s*-\s*$", pd.NA, regex=True)
        df = df.dropna(how="all").drop_duplicates()
        for col in df.select_dtypes(include=["object", "string"]).columns:
            df[col] = df[col].astype("string").str.strip()
        self.__df = df
        return self

    def preencher_ausentes(self, colunas, valor=None):
        """Preenche ausentes com 'valor' (sem valor: usa a média da coluna)."""
        for col in colunas:
            padrao = valor if valor is not None else self.__df[col].astype(float).mean()
            self.__df[col] = self.__df[col].fillna(padrao)
        return self

    # ---------- filtros (devolvem um DataFrame; o original não muda) ----------
    def filtrar(self, coluna, valor):
        """Coluna igual ao valor (ou dentro de uma lista de valores)."""
        if isinstance(valor, (list, tuple, set)):
            return self.__df[self.__df[coluna].isin(valor)]
        return self.__df[self.__df[coluna] == valor]

    def filtrar_intervalo(self, coluna, minimo, maximo):
        """Valores entre mínimo e máximo (inclusive)."""
        return self.__df[self.__df[coluna].between(minimo, maximo)]

    def filtrar_texto(self, coluna, trecho):
        """Texto que contém o trecho (sem diferenciar maiúsculas)."""
        texto = self.__df[coluna].astype("string")
        return self.__df[texto.str.contains(trecho, case=False, na=False, regex=False)]

    def ordenar(self, coluna, crescente=True):
        return self.__df.sort_values(coluna, ascending=crescente)


# ============================================================
# ANALISADOR (regras da Conitec + indicadores)
# ============================================================
class AnalisadorDados:
    """Responsável por analisar os dados da CONITEC (Covid-19)."""
    # nomes da planilha -> nomes curtos, sem acento nem espaço
    COLUNAS = {
        "Qtd": "qtd",
        "Tecnologias e Diretrizes para COVID - 19": "tecnologia",
        "Demandante": "demandante",
        "Análise inicial da Conitec": "analise_inicial",
        "Consulta Pública": "consulta_publica",
        "Análise Final da Conitec": "analise_final",
        "Decisão do Ministério da Saúde": "decisao",
        "Decisão do Ministério da Saúde - Portarias": "portaria",
        "Notas Técnicas": "nota_tecnica",
        "Despacho": "despacho",
    }

    def __init__(self, dataframe):
        self.__df = dataframe.copy()

    @property
    def df(self):
        return self.__df

    def renomear(self): # USE PRIMEIRO
      df = self.__df.rename(columns=self.COLUNAS)
      df["qtd"] = pd.to_numeric(df["qtd"], errors="coerce")
      df["tecnologia"] = df["tecnologia"].str.replace("³", "≥", regex=False)
      self.__df = df
      return self

    def separar_datas(self):
      df = self.__df
      partes = df["demandante"].str.extract(r"^(.+?) em (\d{2}/\d{2}/\d{4})")
      df["demandante"] = partes[0]
      df["data_demanda"] = pd.to_datetime(partes[1], format="%d/%m/%Y")

      pub = df["portaria"].str.extract(r"Publicada em (\d{2}/\d{2}/\d{4})")[0]
      df["data_publicacao"] = pd.to_datetime(pub, format="%d/%m/%Y")
      df["ano"] = df["data_publicacao"].dt.year
      self.__df = df
      return self

    def novas_colunas(self):
      df = self.__df

      df["tipo"] = df["tecnologia"].str.extract(r"^(Diretriz|Vacina)", expand=False)
      df["tipo"] = df["tipo"].fillna("Tratamento")
      df["decisao_positiva"] = ~df["decisao"].str.startswith("Não")
      df["dias_ate_publicacao"] = (df["data_publicacao"] - df["data_demanda"]).dt.days
      df["tem_nota_tecnica"] = df["nota_tecnica"].notna()
      df["tem_despacho"] = df["despacho"].notna()
      self.__df = df
      return self

    def filtrar(self, coluna, valor):
        return TratadorDados(self.__df).filtrar(coluna, valor)

    def ordenar(self, coluna, crescente=True):
        return TratadorDados(self.__df).ordenar(coluna, crescente)

    def agrupar(self, coluna, valor=None, funcao="mean"):
        """Sem 'valor': conta quantos registros há em cada categoria."""
        if valor is None:
            return self.__df.groupby(coluna).size().reset_index(name="quantidade")
        return self.__df.groupby(coluna)[valor].agg(funcao).reset_index()

    def estatisticas(self, coluna="dias_ate_publicacao"):
        return self.__df[coluna].describe()

    def calcular_indicadores(self):
        """Cria a lista de objetos Indicador a partir do DataFrame."""

        total = len(self.__df)
        tecnologias = self.__df["tecnologia"].nunique()

        favoraveis = (
            self.__df["analise_final"].str.contains("favorável", case=False, na=False)
            & ~self.__df["analise_final"].str.contains("desfavorável", case=False, na=False)
        ).sum()

        nao_incorporadas = self.__df["decisao"].str.contains(
            "não incorporação", case=False, na=False
        ).sum()

        incorporadas = (
            self.__df["decisao"].str.contains("incorporação", case=False, na=False)
            & ~self.__df["decisao"].str.contains("não incorporação", case=False, na=False)
        ).sum()

        return [
            Indicador("Total de registros", int(total), casas=0,
                      descricao="Quantidade de linhas na base filtrada"),
            Indicador("Tecnologias analisadas", int(tecnologias), casas=0,
                      descricao="Tecnologias/diretrizes distintas"),
            Indicador("Análises finais favoráveis", int(favoraveis), casas=0,
                      descricao="Análise final da Conitec favorável"),
            Indicador("Incorporações e manutenções", int(incorporadas), casas=0,
                      descricao="Decisões do MS que mencionam incorporação"),
            Indicador("Não incorporações", int(nao_incorporadas), casas=0,
                      descricao="Decisões do MS de não incorporação"),
        ]

## Carregamento → tratamento → análise

In [57]:
from google.colab import files
import os

print("ENVIE SUA PLANILHA (Excel ou CSV):")
uploaded = files.upload()

if not uploaded:
    raise ValueError("Você precisa enviar uma planilha.")

nome_arquivo = list(uploaded.keys())[0]
extensao = os.path.splitext(nome_arquivo)[1].lower()

# Grava o upload com nome fixo (evita problemas com nomes repetidos "(1)")
caminho = f"dados_carregados{extensao}"
with open(caminho, "wb") as f:
    f.write(uploaded[nome_arquivo])

# Escolhe a fonte conforme o tipo do arquivo (polimorfismo: ambas têm carregar())
if extensao in (".xlsx", ".xls"):
    fonte = FonteXLS("Planilha Conitec", caminho)
elif extensao == ".csv":
    fonte = FonteCSV("Planilha Conitec", caminho)
else:
    raise ValueError(f"Formato não suportado: {extensao}. Envie .xlsx, .xls ou .csv")

# ---------- 1) DADOS BRUTOS ----------
df_bruto = fonte.carregar()
df_bruto.columns = df_bruto.columns.astype(str).str.strip()
print(f"Fonte: {fonte.nome} ({type(fonte).__name__}) - {len(df_bruto)} linhas brutas")

faltantes = [c for c in AnalisadorDados.COLUNAS if c not in df_bruto.columns]
if faltantes:
    raise ValueError(
        "Colunas não encontradas: " + str(faltantes)
        + "\nColunas disponíveis: " + str(df_bruto.columns.tolist())
    )

# ---------- 2) TRATADOR (limpeza genérica) ----------
tratador = TratadorDados(df_bruto).limpar()

# ---------- 3) ANALISADOR (regras da Conitec; renomeia colunas) ----------
analisador = AnalisadorDados(tratador.df).renomear().separar_datas().novas_colunas()

# Nomes curtos das colunas (saída do analisador) usados no restante do notebook
COL_TEC = "tecnologia"
COL_DEM = "demandante"
COL_INICIAL = "analise_inicial"
COL_FINAL = "analise_final"
COL_DECISAO = "decisao"
COL_DATA = "data_demanda"

# Ausentes nas colunas principais viram "Não informado" (aparecem nos gráficos/filtro)
df = TratadorDados(analisador.df).preencher_ausentes(
    [COL_TEC, COL_INICIAL, COL_FINAL, COL_DECISAO], "Não informado"
).df

print("REGISTROS APÓS TRATAMENTO:", len(df))
print("COLUNAS:", len(df.columns))

ENVIE SUA PLANILHA (Excel ou CSV):


Saving dados_carregados.xlsx to dados_carregados (1).xlsx
Fonte: Planilha Conitec (FonteXLS) - 20 linhas brutas
REGISTROS APÓS TRATAMENTO: 19
COLUNAS: 24


## Visualizador (gráficos)

In [58]:
class Visualizador:

    def __init__(self, dataframe, tema="plotly_white"):
        self.__df = dataframe.copy()
        self.tema = tema

    @property
    def dados(self):
        return self.__df.copy()

    def barras(self, coluna, titulo):
        dados = (
            self.__df[coluna]
            .value_counts()
            .reset_index()
        )
        dados.columns = ["Categoria", "Quantidade"]

        fig = px.bar(
            dados,
            x="Categoria",
            y="Quantidade",
            text="Quantidade",
            color="Quantidade",
            title=titulo,
            template=self.tema
        )
        fig.update_layout(
            title_x=0.5,
            xaxis_title="Categorias",
            yaxis_title="Quantidade",
            showlegend=False
        )
        return fig

    def barras_horizontais(self, coluna, titulo):
        dados = self.__df[coluna].value_counts().head(10).reset_index()
        dados.columns = ["Categoria", "Quantidade"]

        fig = px.bar(
            dados,
            x="Quantidade",
            y="Categoria",
            orientation="h",
            text="Quantidade",
            title=titulo,
            template=self.tema
        )
        fig.update_layout(
            title_x=0.5,
            yaxis={"categoryorder": "total ascending"}
        )
        return fig

    def pizza(self, coluna, titulo):
        dados = self.__df[coluna].value_counts().reset_index()
        dados.columns = ["Categoria", "Quantidade"]

        fig = px.pie(
            dados,
            names="Categoria",
            values="Quantidade",
            hole=0.4,
            title=titulo,
            template=self.tema
        )
        fig.update_traces(textinfo="percent+label")
        fig.update_layout(title_x=0.5)
        return fig

    def comparar(self):
        inicial = self.__df[COL_INICIAL].value_counts()
        final = self.__df[COL_FINAL].value_counts()

        categorias = sorted(set(inicial.index) | set(final.index))

        comparacao = pd.DataFrame({
            "Categoria": categorias,
            "Análise inicial": [
                int(inicial.get(c, 0)) for c in categorias
            ],
            "Análise final": [
                int(final.get(c, 0)) for c in categorias
            ]
        })

        fig = go.Figure()

        fig.add_trace(go.Bar(
            x=comparacao["Categoria"],
            y=comparacao["Análise inicial"],
            name="Análise inicial"
        ))

        fig.add_trace(go.Bar(
            x=comparacao["Categoria"],
            y=comparacao["Análise final"],
            name="Análise final"
        ))

        fig.update_layout(
            title="Comparação entre análise inicial e final da Conitec",
            barmode="group",
            template=self.tema,
            title_x=0.5,
            xaxis_title="Resultado",
            yaxis_title="Quantidade"
        )

        return fig

    def treemap(self):
        dados = self.__df[COL_DECISAO].value_counts().reset_index()
        dados.columns = ["Decisão", "Quantidade"]

        return px.treemap(
            dados,
            path=["Decisão"],
            values="Quantidade",
            title="Mapa de distribuição das decisões",
            template=self.tema
        )

    def evolucao(self):
        # usa a coluna data_demanda criada pelo AnalisadorDados
        if COL_DATA not in self.__df.columns:
            return None

        anos = (
            self.__df[COL_DATA]
            .dropna()
            .dt.year
            .value_counts()
            .sort_index()
        )

        if anos.empty:
            return None

        dados = anos.reset_index()
        dados.columns = ["Ano", "Quantidade"]

        fig = px.line(
            dados,
            x="Ano",
            y="Quantidade",
            markers=True,
            title="Quantidade de demandas por ano",
            template=self.tema
        )
        fig.update_layout(title_x=0.5)
        return fig

    def tabela(self):
        return self.__df

## Dashboard (usa `AnalisadorDados.calcular_indicadores`)

In [59]:
class Dashboard:

    def __init__(self, dataframe):
        self.__df = dataframe.copy()
        # os indicadores vêm do analisador (composição)
        self.__analisador = AnalisadorDados(self.__df)

    def indicadores(self):
        """Renderiza os cards usando Indicador.nome e Indicador.formatar()."""

        html = '<div style="display:flex;flex-wrap:wrap;gap:12px;">'

        for ind in self.__analisador.calcular_indicadores():
            html += f"""
            <div title="{ind.descricao}" style="
                flex:1;
                min-width:145px;
                background:#17365D;
                color:white;
                padding:20px;
                border-radius:12px;
                text-align:center;
                margin-bottom:10px;
            ">
                <div style="font-size:13px;">{ind.nome}</div>
                <div style="font-size:29px;font-weight:bold;">
                    {ind.formatar()}
                </div>
            </div>
            """

        html += "</div>"
        display(HTML(html))

    def exibir_kpis(self):
        display(HTML("<h2>Indicadores principais</h2>"))
        self.indicadores()

    def exibir_graficos(self):
        viz = Visualizador(self.__df)

        # GRÁFICO DE BARRAS
        display(HTML("<h2>1. Distribuição das decisões</h2>"))
        display(viz.barras(
            COL_DECISAO,
            "Decisões do Ministério da Saúde"
        ))

        # GRÁFICO DE PIZZA
        display(HTML("<h2>2. Participação das decisões</h2>"))
        display(viz.pizza(
            COL_DECISAO,
            "Percentual de cada tipo de decisão"
        ))

        # COMPARAÇÃO
        display(HTML("<h2>3. Comparação das análises</h2>"))
        display(viz.comparar())

        # TREEMAP
        display(HTML("<h2>4. Visualização adicional - Treemap</h2>"))
        display(viz.treemap())

        # DEMANDANTES
        if COL_DEM in self.__df.columns:
            display(HTML("<h2>5. Principais demandantes</h2>"))
            display(viz.barras_horizontais(
                COL_DEM,
                "Demandantes das tecnologias"
            ))

        # EVOLUÇÃO
        display(HTML("<h2>6. Evolução temporal</h2>"))
        grafico = viz.evolucao()

        if grafico is not None:
            display(grafico)
        else:
            display(HTML(
                "<p>Não foram identificadas datas suficientes "
                "para montar a evolução temporal.</p>"
            ))

    def exibir_tabela(self):
        viz = Visualizador(self.__df)

        display(HTML("<h2>7. Tabela completa de dados</h2>"))
        display(
            viz.tabela().reset_index(drop=True).style
            .set_properties(**{
                "text-align": "left",
                "white-space": "normal"
            })
            .set_table_styles([
                {
                    "selector": "th",
                    "props": [
                        ("background-color", "#17365D"),
                        ("color", "white"),
                        ("padding", "10px")
                    ]
                }
            ])
        )

    def executar(self):
        if self.__df.empty:
            display(HTML("<h3>Nenhum registro encontrado.</h3>"))
            return

        display(HTML("""
        <div style="
        background:#17365D;
        color:white;
        padding:25px;
        border-radius:12px;
        ">
        <h1>PAINEL COVID-19</h1>
        <p>Tecnologias, tratamentos, prevenção e decisões da Conitec</p>
        <p>Projeto de Programação Orientada a Objetos</p>
        </div>
        """))

        self.exibir_kpis()
        self.exibir_graficos()
        self.exibir_tabela()

## Executar o painel

In [60]:
print("DADOS CARREGADOS:", len(df), "registros")
print("COLUNAS:", len(df.columns))

dashboard = Dashboard(df)
dashboard.executar()

DADOS CARREGADOS: 19 registros
COLUNAS: 24


,qtd,tecnologia,demandante,analise_inicial,Relatório de Recomendação inicial,Relatórios para a Sociedade,consulta_publica,Contribuições de experiência e opinião,Contribuições técnico - científicas,analise_final,Relatório de recomendação Final,decisao,Decisão do Ministério da Saúde - Relatório,portaria,nota_tecnica,despacho,data_demanda,data_publicacao,ano,tipo,decisao_positiva,dias_ate_publicacao,tem_nota_tecnica,tem_despacho
0,15,"Nirmatrelvir/Ritonavir para o tratamento da Covid-19 para pacientes com sintomas leves a moderados, que não requerem oxigênio suplementar, independentemente do status vacinal e com idade ≥ 65 anos ou imunocomprometidos com idade ≥ 18 anos","Secretaria de Ciência, Tecnologia e Inovação e do Complexo Econômico-Industrial da Saúde",Favorável à manutenção da incorporação,Relatório de recomendação inicial,Relatório para a Sociedade,Realizada de 26 a 15/01/2024,Experiêcia e opinião,Técnico e científico,Favorável à manutenção da incorporação,Relatório de recomendação Final,Manutenção da incorporação,Relatório com decisão,Portaria SCTIE/MS nº 9 - Publicada em 29/05/2024,,,2024-03-05 00:00:00,2024-05-29 00:00:00,2024,Tratamento,True,85,False,False
1,14,Vacina Pfizer/BioNTech para a imunização ativa de indivíduos na faixa etária de 6 meses a 5 anos incompletos para a prevenção da Covid-19,Secretaria de Vigilância em Saúde,Favorável à incorporação,Relatório de recomendação inicial,Relatório para a Sociedade,Realizada de 06 a 15/12/2022,Experiêcia e opinião,Técnico e científico,Favorável à incorporação,Relatório de recomendação Final,Incorporação,Relatório com decisão,Portaria SCTIE/MS nº 181 - Publicada em 29/12/2022,,,2022-10-18 00:00:00,2022-12-29 00:00:00,2022,Vacina,True,72,False,False
2,13,Molnupiravir para pacientes infectados por SARS-CoV-2 não hospitalizados com alto risco de agravamento da doença,Secretaria Executiva,Desfavorável à incorporação,Relatório de recomendação inicial,Relatório para a Sociedade,Realizada de 27/06/2022 a 18/07/2022,Experiêcia e opinião,Técnico e científico,Desfavorável à incorporação,Relatório de recomendação Final,Não incorporação,Relatório com decisão,Portaria SCTIE/MS nº 102 - Publicada em 12/09/2022,,,2022-05-16 00:00:00,2022-09-12 00:00:00,2022,Tratamento,False,119,False,False
3,12,Tocilizumabe para tratamento de pacientes adultos com Covid-19 hospitalizados,Ministro da Saúde,Favorável à incorporação,Relatório de recomendação inicial,Relatório para a Sociedade,Realizada de 13/07/2022 a 01/08/2022,Experiêcia e opinião,Técnico e científico,Favorável à incorporação,Relatório de recomendação Final,Incorporação,Relatório com decisão,Portaria SCTIE/MS nº 101 - Publicada em 12/09/2022,,,2022-04-14 00:00:00,2022-09-12 00:00:00,2022,Tratamento,True,151,False,False
4,11,"Cilgavimabe + tixagevimabe para profilaxia pré-exposição à Covid-19, em indivíduos adultos e pediátricos (com 12 anos de idade ou mais, pesando pelo menos 40 kg)",Secretaria Executiva,Desfavorável à incorporação,Relatório de recomendação inicial,Relatório para a Sociedade,Realizada de 08 a 27/07/2022,Experiêcia e opinião,Técnico e científico,Desfavorável à incorporação,Relatório de recomendação Final,Não incorporação,Relatório com decisão,Portaria SCTIE/MS nº 115 - Publicada em 29/09/2022,,,2022-04-11 00:00:00,2022-09-29 00:00:00,2022,Tratamento,False,171,False,False
5,10,Nirmatrelvir + ritonavir para o tratamento da Covid-19 em adultos que não requerem oxigênio suplementar e que apresentam risco aumentado de progressão para Covid-19 grave,Secretaria Executiva,Favorável à incorporação,Relatório de recomendação inicial,Relatório para a Sociedade,Realizada de 18 a 27/04/2022,Experiêcia e opinião,Técnico e científico,Favorável à incorporação,Relatório de recomendação Final,Incorporação,Relatório com decisão,Portaria SCTIE/MS nº 44 - Publicada em 06/05/2022,,,2022-04-05 00:00:00,2022-05-06 00:00:00,2022,Tratamento,True,31,False,False
6,9,"Baricitinibe para tratamento da Covid-19 em pacientes adultos, hospitalizados e 

## Painel com filtro interativo

In [61]:
from google.colab import output
output.enable_custom_widget_manager()

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

In [62]:
assert "df" in globals(), "A variável df não foi encontrada."
assert "Dashboard" in globals(), "Execute primeiro as classes do projeto."


# COL_DECISAO já foi definida na célula de carregamento (nome curto: "decisao")

if COL_DECISAO not in df.columns:
    raise ValueError(
        f"Coluna não encontrada. Colunas disponíveis: {df.columns.tolist()}"
    )

opcoes = ["Todas"] + sorted(
    df[COL_DECISAO]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)


filtro_decisao = widgets.Dropdown(
    options=opcoes,
    value="Todas",
    description="Decisão:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="95%")
)

botao_aplicar = widgets.Button(
    description="APLICAR FILTRO",
    button_style="primary",
    icon="filter",
    layout=widgets.Layout(width="200px", height="40px")
)

botao_limpar = widgets.Button(
    description="LIMPAR FILTRO",
    button_style="",
    icon="refresh",
    layout=widgets.Layout(width="200px", height="40px")
)

resultado_painel = widgets.Output()


def atualizar_painel():

    with resultado_painel:
        clear_output(wait=True)

        try:
            if filtro_decisao.value == "Todas":
                dados_filtrados = df.copy()
            else:
                dados_filtrados = df[
                    df[COL_DECISAO].astype(str)
                    == filtro_decisao.value
                ].copy()

            print(
                "Quantidade de registros encontrados:",
                len(dados_filtrados)
            )

            if dados_filtrados.empty:
                print("Nenhum registro encontrado.")
                return


            painel_filtrado = Dashboard(dados_filtrados)


            painel_filtrado.executar()

        except Exception as erro:
            print("ERRO AO ATUALIZAR O VISUALIZADOR:")
            print(type(erro).__name__, ":", erro)

            import traceback
            traceback.print_exc()


def clicar_aplicar(botao):
    atualizar_painel()


def clicar_limpar(botao):
    filtro_decisao.value = "Todas"
    atualizar_painel()

botao_aplicar.on_click(clicar_aplicar)
botao_limpar.on_click(clicar_limpar)


display(HTML("""
<div style="
    background:#05a8e8;
    color:white;
    padding:20px;
    border-radius:12px;
    margin-top:20px;
">
<h2>PAINEL DE FILTROS</h2>
<p>Selecione uma decisão e atualize as visualizações.</p>
</div>
"""))

display(filtro_decisao)

display(
    widgets.HBox([
        botao_aplicar,
        botao_limpar
    ])
)

display(resultado_painel)

atualizar_painel()

Dropdown(description='Decisão:', layout=Layout(width='95%'), options=('Todas', 'Aprovação', 'Aprovação com sol…

Output()